# 🧠 Memory Observatory — LangGraph × Mem0 × OpenRouter

**Class project:** Maya's AI Learning Buddy. Each chat turn renders all **seven memory categories**, the **real operations performed**, and the updated memory state. Includes external lookup and a prospective task registry (not a notification scheduler).

**Required keys:** [OpenRouter](https://openrouter.ai/keys) + [Mem0 Platform](https://app.mem0.ai/). OpenRouter powers chat; Mem0 Cloud manages long-term memory (including its own extraction/embedding infrastructure). Charges/rate limits depend on your services.

> Run cells from top to bottom. This notebook has not been integration-tested with your credentials. Mem0 Cloud changes its API periodically; consult its current docs if a signature changes.

In [ ]:
%pip -q install -U langgraph langchain-openai mem0ai ipywidgets requests

In [ ]:
import os
from getpass import getpass
os.environ["OPEN_ROUTER_KEY"] = getpass("OpenRouter API key: ")
os.environ["MEM0_API_KEY"] = getpass("Mem0 Platform API key: ")
OPENROUTER_MODEL = "openrouter/free"  # Replace with a model available in your OpenRouter account
print("Keys configured (not displayed).")

## Architecture

`USER → recall (Mem0 search) → respond (OpenRouter LLM) → classify & consolidate (Mem0 + registries) → DASHBOARD`

- **Parametric:** model provenance — the model's weights cannot be inspected or updated here.
- **Working:** LangGraph thread checkpointer.
- **Semantic & episodic:** Mem0 facts/events, classified for display. These are *application-level labels*, not separate Mem0 engines.
- **Procedural:** stored preferences/rules from explicit user instructions.
- **External:** live public API lookup, cached with timestamp.
- **Prospective:** pending tasks recorded locally; **no background reminder delivery**.

Read/write status is backed by executed application operations, not guessed activity.

In [ ]:
import json, re, html, requests
from datetime import datetime, timedelta, timezone
from zoneinfo import ZoneInfo
from collections import defaultdict
from typing import Annotated, TypedDict
from IPython.display import HTML, display
from langchain_openai import ChatOpenAI
from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver
from mem0 import MemoryClient

llm = ChatOpenAI(
    model=OPENROUTER_MODEL,
    api_key=os.environ["OPEN_ROUTER_KEY"],
    base_url="https://openrouter.ai/api/v1",
    temperature=0.2,
    max_retries=2,
)
mem0 = MemoryClient(api_key=os.environ["MEM0_API_KEY"])
checkpointer = InMemorySaver()

# Local auxiliary registries for teaching purposes. Restarting Colab clears them.
registry = defaultdict(lambda: {"procedural": [], "prospective": [], "external": [], "episodic": []})
operations = []
turn_counts = defaultdict(int)
USER = "maya_colab_demo"  # Customize to prevent mixing with other students.
THREAD = "lesson_1"
TZ = ZoneInfo("Asia/Kolkata")

def log(kind, action, detail):
    operations.append({"type": kind, "action": action, "detail": str(detail)[:240]})

def unpack(result):
    if isinstance(result, dict): return result.get("results", result.get("memories", []))
    if isinstance(result, list): return result
    return []

def label_memory(item):
    # Labels are heuristics used by this class demo (not Mem0 native memory types).
    text = str(item.get("memory", "")).lower()
    if any(w in text for w in ["yesterday", "last week", "attended", "struggled", "completed", "passed", "first class", "studied"]):
        return "episodic"
    if any(w in text for w in ["always", "prefer", "whenever", "explain using", "avoid jargon"]):
        return "procedural"
    return "semantic"

def mem_search(query, user):
    result = mem0.search(query, filters={"user_id": user})
    items = unpack(result)
    log("semantic", "READ", f"Mem0 search: {len(items)} matching records")
    return items

def mem_all(user):
    result = mem0.get_all(filters={"user_id": user})
    return unpack(result)


In [ ]:
class MemoryState(TypedDict, total=False):
    messages: Annotated[list, add_messages]
    user_id: str
    thread_id: str
    recalled: list
    external_context: str
    answer: str


def recall_node(state: MemoryState):
    user = state["user_id"]
    question = next((m.content for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), "")
    hits = mem_search(question + " preferences learning goals background experience", user)
    log("working", "READ", f"{len(state['messages'])} thread messages in LangGraph state")
    prefs = registry[user]["procedural"]
    if prefs: log("procedural", "READ", f"{len(prefs)} explicit rule(s)")
    tasks = registry[user]["prospective"]
    if tasks: log("prospective", "READ", f"{len(tasks)} pending task(s)")
    return {"recalled": hits}


def external_node(state: MemoryState):
    question = next((m.content for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), "")
    if not re.search(r"(latest|current|today|right now).*python version|python version.*(latest|current)", question, re.I):
        return {"external_context": ""}
    try:
        # Query the official Python downloads RSS feed.
        import xml.etree.ElementTree as ET
        feed = requests.get("https://www.python.org/downloads/feed.rss", timeout=10)
        feed.raise_for_status()
        root = ET.fromstring(feed.content)
        items = root.findall(".//item")
        title = items[0].findtext("title") if items else "No releases found"
        link = items[0].findtext("link") if items else "https://www.python.org/downloads/"
        entry = {"query": question, "value": title, "source": link,
                 "at": datetime.now(timezone.utc).isoformat()}
        registry[state["user_id"]]["external"].append(entry)
        log("external", "WRITE", title)
        return {"external_context": f"Python downloads RSS latest entry: {title}. Source: {link}. This might not be the latest stable release; verify on python.org."}
    except Exception as e:
        log("external", "ERROR", f"Lookup failed: {type(e).__name__}: {e}")
        return {"external_context": "Live lookup failed. Do not invent a current version."}


def respond_node(state: MemoryState):
    hits = [i.get("memory", "") for i in state.get("recalled", []) if isinstance(i, dict)]
    rules = registry[state["user_id"]]["procedural"]
    system = f"""You are Maya's friendly AI Learning Buddy. Be clear, accurate, and brief.
Relevant memories (may be stale): {json.dumps(hits[:10])}
User's explicit instructions: {json.dumps(rules[-6:])}
External tool result: {state.get('external_context', '')}
For reminders: say you can record a pending task in this notebook, but cannot proactively notify the user.
Do not claim a tool call succeeded unless the results say it did.
"""
    # Bound context length. Long-term recall comes from Mem0 instead.
    reply = llm.invoke([SystemMessage(content=system)] + state["messages"][-12:])
    log("parametric", "USED", f"Model {OPENROUTER_MODEL} called (weights not accessible)")
    log("working", "WRITE", "Assistant reply checkpointed in current thread")
    return {"messages": [reply], "answer": reply.content}


def consolidate_node(state: MemoryState):
    user = state["user_id"]
    question = next((m.content for m in reversed(state["messages"]) if isinstance(m, HumanMessage)), "")
    low = question.lower()
    # Explicit procedural preferences are tracked separately for accurate display.
    if re.search(r"(whenever|always|from now on|i prefer you|please explain using)", low):
        if question not in registry[user]["procedural"]:
            registry[user]["procedural"].append(question)
            log("procedural", "WRITE", question)
    # Simple event log: events are observed facts about this turn, not verified timelines.
    if re.search(r"(yesterday|last week|attended|studied|struggled|mastered|completed)", low):
        event = {"text": question, "recorded_at": datetime.now(TZ).isoformat()}
        registry[user]["episodic"].append(event)
        log("episodic", "WRITE", question)
    if re.search(r"remind me", low):
        match = re.search(r"(remind me .*)", question, re.I)
        task = {"text": match.group(1) if match else question,
                "created": datetime.now(TZ).isoformat(), "status": "PENDING / NOT SCHEDULED"}
        registry[user]["prospective"].append(task)
        log("prospective", "WRITE", task["text"])
    # Mem0 decides if the user utterance contains a useful memory.
    before = mem_all(user)
    result = mem0.add([{"role": "user", "content": question}], user_id=user,
                      metadata={"source": "memory_observatory"})
    after = mem_all(user)
    before_ids = {i.get("id") for i in before if isinstance(i, dict)}
    added = [i.get("memory", "") for i in after if isinstance(i, dict) and i.get("id") not in before_ids]
    changed = [i.get("memory", "") for i in after if isinstance(i, dict) and i.get("id") in before_ids and i not in before]
    log("semantic", "ADD_CALL", f"Mem0 processed message; +{len(added)} new, {len(changed)} changed, before={len(before)}, after={len(after)}")
    for item in after:
        if isinstance(item, dict) and (item.get("memory", "") in added + changed):
            log(label_memory(item), "WRITE", item.get("memory", ""))
    return {}

builder = StateGraph(MemoryState)
builder.add_node("recall", recall_node)
builder.add_node("external", external_node)
builder.add_node("respond", respond_node)
builder.add_node("consolidate", consolidate_node)
builder.add_edge(START, "recall")
builder.add_edge("recall", "external")
builder.add_edge("external", "respond")
builder.add_edge("respond", "consolidate")
builder.add_edge("consolidate", END)
graph = builder.compile(checkpointer=checkpointer)
print("Graph compiled ✓")

## 🎨 Seven-panel Memory Observatory

A dashboard renders after **every** `chat()` call. Cards show latest observed activity plus state. For Mem0, the stored facts are displayed in semantic/episodic/procedural buckets using transparent heuristics. A `READ` means the application executed a read; it does *not* prove the model used that memory in its reasoning.

In [ ]:
THEMES = {
    "parametric": ("◈", "Model knowledge", "#3b82f6"),
    "working": ("◎", "Conversation buffer", "#6366f1"),
    "semantic": ("✦", "Facts & goals", "#8b5cf6"),
    "episodic": ("◷", "Past experiences", "#a855f7"),
    "procedural": ("⚙", "Behavior rules", "#ec4899"),
    "external": ("↗", "Fresh tool results", "#14b8a6"),
    "prospective": ("◇", "Future intentions", "#f59e0b"),
}

def dashboard(user, thread, question, answer, state, ops):
    facts = mem_all(user)
    buckets = {k: [] for k in THEMES}
    for f in facts:
        if isinstance(f, dict): buckets[label_memory(f)].append(f.get("memory", ""))
    buckets["parametric"] = [f"OpenRouter: {OPENROUTER_MODEL}", "Model weights are not stored in agent state"]
    buckets["working"] = [f"{len(state.get('messages', []))} messages in thread {thread}"] + [
        ("You: " if isinstance(m, HumanMessage) else "AI: ") + str(m.content)[:130]
        for m in state.get("messages", [])[-4:]
    ]
    for p in registry[user]["procedural"]:
        if p not in buckets["procedural"]: buckets["procedural"].append(p)
    for e in registry[user]["episodic"][-8:]:
        if e["text"] not in buckets["episodic"]: buckets["episodic"].append(e["text"])
    buckets["external"] = [f"{x['value']} · {x['source']}" for x in registry[user]["external"][-6:]]
    buckets["prospective"] = [f"{x['text']} — {x['status']}" for x in registry[user]["prospective"][-8:]]
    cards = []
    for kind, (symbol, subtitle, color) in THEMES.items():
        relevant = [o for o in ops if o["type"] == kind]
        actions = " · ".join(sorted(set(o["action"] for o in relevant))) or "NO OPERATION"
        items = "".join(f'<div class="memitem">{html.escape(str(v))}</div>' for v in buckets[kind][-7:]) or '<div class="empty">No stored items</div>'
        cards.append(f"""<section class="card" style="--tone:{color}"><div class="cardhead"><span class="icon">{symbol}</span><div><h3>{kind.title()}</h3><small>{subtitle}</small></div><span class="count">{len(buckets[kind])}</span></div><div class="tag">{html.escape(actions)}</div><div class="items">{items}</div></section>""")
    event_rows = "".join(f'<div class="event"><span class="eventtype">{html.escape(o["type"].upper())}</span><b>{html.escape(o["action"])}</b> {html.escape(o["detail"])}</div>' for o in ops)
    markup = f"""<style>
    .memwrap{{font-family:Inter,ui-sans-serif,system-ui,sans-serif;background:linear-gradient(145deg,#0b1024,#191533);color:#eef2ff;padding:28px;border-radius:20px;line-height:1.45}}
    .memwrap *{{box-sizing:border-box}} .memtitle{{font-weight:900;letter-spacing:-.06em;font-size:32px;margin:0}}
    .memsub{{color:#b9bad5;font-size:13px;margin:7px 0 21px}}
    .chatbubble{{background:#202747;border:1px solid #364165;padding:13px 16px;margin:8px 0;border-radius:12px;white-space:pre-wrap;overflow-wrap:anywhere}}
    .chatbubble.ai{{background:#162d33;border-color:#245d5c}}.speaker{{font-size:11px;letter-spacing:.11em;color:#a9b7df;font-weight:800;margin-bottom:4px}}
    .grid{{display:grid;grid-template-columns:repeat(auto-fit,minmax(245px,1fr));gap:13px;margin-top:23px}}
    .card{{background:#1f2440;border:1px solid #333a60;border-top:3px solid var(--tone);border-radius:14px;padding:15px;min-height:170px}}
    .cardhead{{display:flex;align-items:center;gap:10px}} .icon{{font-size:26px;color:var(--tone);width:25px}}h3{{font-size:16px;margin:0}}
    small{{color:#a9adca}}.count{{margin-left:auto;padding:3px 9px;border-radius:12px;background:#32385c;font-size:12px}}
    .tag{{display:inline-block;font-weight:750;font-size:10px;letter-spacing:.07em;color:#d2cbff;background:#34325c;padding:5px 8px;border-radius:6px;margin:12px 0}}
    .memitem{{background:#292f4d;padding:8px 10px;margin-bottom:6px;border-radius:8px;font-size:12px;overflow-wrap:anywhere}}.empty{{font-size:12px;color:#858aaa}}
    .eventbox{{margin-top:22px;border:1px solid #333a60;border-radius:14px;padding:15px}}.event{{padding:7px 1px;border-bottom:1px solid #303652;font-size:12px;color:#bcc7dd}}
    .event:last-child{{border:none}}.eventtype{{color:#93c5fd;min-width:94px;display:inline-block;font-size:10px;font-weight:800}}.event b{{color:#fff;margin-right:7px}}
    </style><div class="memwrap"><h1 class="memtitle">◉ Memory Observatory</h1><div class="memsub">LIVE MEMORY STATE · USER {html.escape(user)} · THREAD {html.escape(thread)} · TURN {turn_counts[(user,thread)]}</div>
    <div class="chatbubble"><div class="speaker">YOU ASKED</div>{html.escape(question)}</div>
    <div class="chatbubble ai"><div class="speaker">AI LEARNING BUDDY</div>{html.escape(str(answer))}</div>
    <div class="grid">{''.join(cards)}</div><div class="eventbox"><b>⚡ Actual operation trace · {len(ops)} events</b>{event_rows}</div>
    <p style="font-size:11px;color:#9ca3af">Note: labels are educational classifications; statuses denote application calls, not proof of attention. Pending tasks are NOT scheduled notifications.</p></div>"""
    display(HTML(markup))


def chat(question, user=USER, thread=THREAD):
    operations.clear()
    config = {"configurable": {"thread_id": f"{user}::{thread}"}}
    turn_counts[(user, thread)] += 1
    result = graph.invoke({"messages": [HumanMessage(content=question)],
                           "user_id": user, "thread_id": thread}, config=config)
    dashboard(user, thread, question, result.get("answer", ""), result, list(operations))
    return result.get("answer", "")

print("Dashboard ready ✓")

## ▶️ Try these prompts one at a time

Start with the first turn below. Every `chat()` invocation displays the seven-panel dashboard. Run later examples interactively to make Mem0 memory evolution visible. **Do not run all requests at once**—Mem0 may take seconds per operation and incur usage charges.

In [ ]:
# Define fastapi wrapping the chat function and log the input, output JSON for telemetry

import json
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import Optional
from fastapi.testclient import TestClient

# 1. Initialize FastAPI app
app = FastAPI(title="Maya's AI Learning Buddy API", version="1.0")

# 2. Define Request and Response Schemas
class ChatRequest(BaseModel):
    question: str
    user: Optional[str] = USER
    thread: Optional[str] = THREAD

class ChatResponse(BaseModel):
    user: str
    thread: str
    question: str
    answer: str

# 3. Create the endpoint and log input/output JSON for telemetry
@app.post("/chat", response_model=ChatResponse)
def api_chat(payload: ChatRequest):
    # Telemetry: Log incoming request JSON
    input_json = payload.model_dump()
    print("\n=== [TELEMETRY] INCOMING REQUEST ===")
    print(json.dumps(input_json, indent=2))

    try:
        # Call the existing chat function in our notebook context
        response_text = chat(
            question=payload.question,
            user=payload.user,
            thread=payload.thread
        )
    except Exception as e:
        print(f"[TELEMETRY] ERROR processing request: {str(e)}")
        raise HTTPException(status_code=500, detail=str(e))

    output_json = {
        "user": payload.user,
        "thread": payload.thread,
        "question": payload.question,
        "answer": response_text
    }

    # Telemetry: Log outgoing response JSON
    print("\n=== [TELEMETRY] OUTGOING RESPONSE ===")
    print(json.dumps(output_json, indent=2))

    return output_json

# 4. Local verification using TestClient
print("FastAPI app created. Testing locally via TestClient...")
client = TestClient(app)

test_payload = {
    "question": "What are we learning today?",
    "user": USER,
    "thread": THREAD
}

response = client.post("/chat", json=test_payload)


In [ ]:
chat("Hi! I'm Maya, a software engineer learning Python for AI engineering.")

In [ ]:
# Uncomment and run ONE line at a time:
chat("Yesterday I struggled with nested Python loops in my class.")
chat("Whenever you teach me, use everyday examples and avoid jargon.")
chat("Explain Python functions to me.")
chat("What did I struggle with yesterday?")
chat("What is the latest Python version right now?")
chat("Remind me tomorrow to practice nested loops.")
chat("I have now mastered nested loops!")
chat("What do you remember about me?", thread="new_conversation")
chat("What do you know about me?", user="different_student", thread="test")

In [ ]:
chat("Remind me tomorrow at 7 PM to practice Python nested loops.")

## 🧪 What to explain in class

1. **Seven visible cards ≠ seven invoked memories:** each card displays its contents; only the trace says what actually ran.
2. **LangGraph short-term memory vs. Mem0 long-term memory:** new `thread_id` clears the conversation context, but Mem0 can still recall user facts from other threads.
3. **Memory correctness is not automatic:** facts can be incorrectly extracted, repeated, stale or misclassified. The classification regex and task parser are transparent teaching shortcuts, not reliable production extractors.
4. **Memory lifecycle:** `search` → prompt augmentation → model response → Mem0 add → display updated state.
5. **Colab runtime restart:** LangGraph checkpointer and local registries vanish. Mem0 Cloud can retain its own records. Use a persistent database, proper scheduler, time parsing, permissions, and deletion controls for production.
6. **Security:** never store API keys or sensitive personal data in notebook output. Mem0 operations send user messages to a hosted provider.

### References
- [Mem0 search API](https://docs.mem0.ai/core-concepts/memory-operations/search)
- [Mem0 quickstart](https://docs.mem0.ai/platform/quickstart)
- [LangGraph persistence](https://docs.langchain.com/oss/python/langgraph/persistence)
- [OpenRouter API](https://openrouter.ai/docs/quickstart)
